In [12]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available. Select an A100 GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")
print(subprocess.check_output(["nvidia-smi"], text=True))

if "A100" not in gpu_name.upper():
    raise RuntimeError(f"Expected an A100 GPU, but Colab assigned: {gpu_name}")

print("A100 GPU verified.")

GPU: NVIDIA A100-SXM4-40GB
Thu Sep 17 01:27:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             43W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+--------------------

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
!nvidia-smi
!pip install ultralytics
!pip install --upgrade sympy # fixes crash in yolo model creation
import ultralytics
ultralytics.checks()


Ultralytics 8.4.154 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 49.9/112.6 GB disk)


In [15]:
%%bash
mkdir -p zips dataset
for dataset in dataset-ACANAT24 \
dataset-ACANAT26 \
dataset-ACAT24 \
dataset-ACAT25 \
dataset-basalpha1 \
dataset-Cascadia-FRI \
dataset-cascadia-SUN \
dataset-hocr25-f-fri \
dataset-hocr25-f-satam \
dataset-hocr25-f-satpm \
dataset-hocr25-s-fri \
dataset-hocr25-s-satam \
dataset-hocr25-s-satpm \
dataset-hocr25-s-sunam \
dataset-negatives \
dataset-PRR24 \
dataset-sdcc24 \
dataset-sdcc25 \
dataset-sdcc25-sunam \
  ; do
  if [ ! -f "zips/$dataset.zip" ]; then
    echo "downloading $dataset.zip"
    if ! wget -q -O "zips/$dataset.zip" "https://storage.googleapis.com/training.crewtimer.com/datasets/$dataset.zip"; then
      echo "Failed to download $dataset"
      rm -f "zips/$dataset.zip"
    else
       # unzip -q "zips/$dataset.zip" -d dataset
       echo "zip file $dataset.zip downloaded"
    fi
  fi
done


downloading dataset-Cascadia-FRI.zip
zip file dataset-Cascadia-FRI.zip downloaded
downloading dataset-cascadia-SUN.zip
zip file dataset-cascadia-SUN.zip downloaded
downloading dataset-hocr25-f-fri.zip
zip file dataset-hocr25-f-fri.zip downloaded
downloading dataset-hocr25-f-satam.zip
zip file dataset-hocr25-f-satam.zip downloaded
downloading dataset-hocr25-f-satpm.zip
zip file dataset-hocr25-f-satpm.zip downloaded
downloading dataset-hocr25-s-fri.zip
zip file dataset-hocr25-s-fri.zip downloaded
downloading dataset-hocr25-s-satam.zip
zip file dataset-hocr25-s-satam.zip downloaded
downloading dataset-hocr25-s-satpm.zip
zip file dataset-hocr25-s-satpm.zip downloaded
downloading dataset-hocr25-s-sunam.zip
zip file dataset-hocr25-s-sunam.zip downloaded
downloading dataset-negatives.zip
zip file dataset-negatives.zip downloaded
downloading dataset-PRR24.zip
zip file dataset-PRR24.zip downloaded
downloading dataset-sdcc24.zip
zip file dataset-sdcc24.zip downloaded
downloading dataset-sdcc25.z

In [16]:
!ls -l dataset zips
#!rm -rf zips

dataset:
total 148
drwxr-xr-x 2 root root 32768 Sep 17 01:26 card-labels
drwxr-xr-x 2 root root 57344 Sep 17 01:26 images
drwxr-xr-x 2 root root 61440 Sep 17 01:26 labels

zips:
total 8445408
-rw-r--r-- 1 root root  465847474 Sep 16 23:24 dataset-ACANAT24.zip
-rw-r--r-- 1 root root  454309345 Sep 16 23:24 dataset-ACANAT26.zip
-rw-r--r-- 1 root root  269473431 Sep 16 23:22 dataset-ACAT24.zip
-rw-r--r-- 1 root root   67133926 Sep 16 23:22 dataset-ACAT25.zip
-rw-r--r-- 1 root root   41943040 Sep 17 01:13 dataset-basalpha1.zip
-rw-r--r-- 1 root root  175835922 Sep 16 23:25 dataset-Cascadia-FRI.zip
-rw-r--r-- 1 root root  248632332 Sep 16 23:26 dataset-cascadia-SUN.zip
-rw-r--r-- 1 root root  351127925 Sep 16 23:27 dataset-hocr25-f-fri.zip
-rw-r--r-- 1 root root 2235342937 Sep 16 23:45 dataset-hocr25-f-satam.zip
-rw-r--r-- 1 root root        280 Sep 16 23:26 dataset-hocr25-f-satpm.zip
-rw-r--r-- 1 root root  368275184 Sep 16 23:30 dataset-hocr25-s-fri.zip
-rw-r--r-- 1 root root  509549443 S

In [17]:
%%bash
rm -rf dataset tmp-dataset
mkdir -p dataset tmp-dataset
for file in zips/*; do
  unzip -q $file -d tmp-dataset
done
for dir in images labels card-labels; do
  mkdir -p dataset/$dir
  mv tmp-dataset/*/$dir/* dataset/$dir
  echo "$dir: `ls dataset/$dir | wc -l`"
done
rm -rf tmp-dataset

images: 4911
labels: 4911
card-labels: 3128


[zips/dataset-basalpha1.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of zips/dataset-basalpha1.zip or
        zips/dataset-basalpha1.zip.zip, and cannot find zips/dataset-basalpha1.zip.ZIP, period.


In [18]:
%%bash
mv dataset/labels dataset/labels-orig
# rm -f dataset/*/ACA*
rm -f dataset/*/*.onnx
ls dataset/images |wc -l
ls dataset/labels-orig |wc -l
# look for mismatched labels and images filenames
comm -3 <(find dataset/images -type f -printf '%f\n' | sed 's/\.[^.]*$//' | sort) <(find dataset/labels-orig -type f -printf '%f\n' | sed 's/\.[^.]*$//' | sort)

4911
4911


In [19]:
import os, glob

# Normalize labels for detection.  Remove keypoint data.
!rm -rf dataset/labels
!mkdir -p dataset/labels

# Root of your dataset (match data.yaml:path)
ROOT = "/content/dataset"
IN_LABELS  = os.path.join(ROOT, "labels-orig")
OUT_LABELS = os.path.join(ROOT, "labels")
os.makedirs(OUT_LABELS, exist_ok=True)

# For detection, we expect 5 values: class, cx, cy, w, h
EXPECTED = 5

def clamp01(v): return max(0.0, min(1.0, v))

fixed, skipped, bad = 0, 0, []

for in_path in glob.glob(os.path.join(IN_LABELS, "**/*.txt"), recursive=True):
    rel = os.path.relpath(in_path, IN_LABELS)
    out_path = os.path.join(OUT_LABELS, rel)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    new_lines = []
    with open(in_path, "r") as f:
        for ln, raw in enumerate(f, 1):
            s = raw.strip()
            if not s:
                continue
            parts = s.split()

            # Coerce tokens to float-friendly strings
            try:
                nums = [float(p) for p in parts]
            except ValueError:
                bad.append((in_path, ln, "non-numeric token(s)"))
                continue

            # For detection, we only care about the first 5 values (class, cx, cy, w, h)
            if len(nums) < EXPECTED:
                skipped += 1
                bad.append((in_path, ln, f"len={len(nums)} (expected {EXPECTED})"))
                continue

            nums = nums[:EXPECTED] # Keep only the first 5 values

            # Unpack
            cls = int(nums[0])
            cx, cy, w, h = nums[1:5]

            # Clamp normalized coords
            cx = clamp01(cx); cy = clamp01(cy)
            w  = max(0.0, min(1.0, w))
            h  = max(0.0, min(1.0, h))

            new_lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
            fixed += 1

    if new_lines:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        with open(out_path, "w") as g:
            g.writelines(new_lines)

print(f"Fixed/rewrote lines: {fixed}")
print(f"Skipped malformed lines: {skipped}")
if bad:
    print("Examples of issues:")
    for b in bad[:20]:
        print(b)
    if len(bad) > 20:
        print(f"... and {len(bad)-20} more")

Fixed/rewrote lines: 6764
Skipped malformed lines: 0


In [20]:
#
# Create training and validation datasets
#
!rm -rf dataset/train dataset/val
!find /content/dataset -name "*.cache" -delete
import os
import random

# Set random seed for reproducibility
random.seed(42)

# Define paths
base_dir = "/content/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

train_images_dir = os.path.join(base_dir, "train/images")
train_labels_dir = os.path.join(base_dir, "train/labels")
val_images_dir   = os.path.join(base_dir, "val/images")
val_labels_dir   = os.path.join(base_dir, "val/labels")

# Create target directories
for d in [train_images_dir, train_labels_dir, val_images_dir, val_labels_dir]:
    os.makedirs(d, exist_ok=True)

# List and shuffle images
all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.png'))]
random.shuffle(all_images)

# Split 80% train / 20% val
split_idx = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

# Link images and labels
def link_files(image_list, target_img_dir, target_lbl_dir):
    for img_name in image_list:
        label_name = os.path.splitext(img_name)[0] + '.txt'
        src_img = os.path.join(images_dir, img_name)
        src_lbl = os.path.join(labels_dir, label_name)
        dst_img = os.path.join(target_img_dir, img_name)
        dst_lbl = os.path.join(target_lbl_dir, label_name)

        # Remove existing links/files if present
        for src, dst in [(src_img, dst_img), (src_lbl, dst_lbl)]:
            if os.path.exists(dst):
                os.remove(dst)
            os.symlink(src, dst)

link_files(train_images, train_images_dir, train_labels_dir)
link_files(val_images, val_images_dir, val_labels_dir)


In [ ]:
# Set data.yaml for non pose pt file such as yolov8m.pt
data_yaml = """
path: /content/dataset
train: train/images
val: val/images

names:
  0: boat
"""

# Ensure clean data.yaml by removing any previous version before writing
!rm -f /content/dataset/data.yaml
with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

import threading
from ultralytics import YOLO
from datetime import datetime
from zoneinfo import ZoneInfo # Import ZoneInfo for timezone awareness

# Generate a timestamped name: train-YYYYMMDD_HHMM
timestamp = datetime.now(ZoneInfo('America/Los_Angeles')).strftime('train-%Y-%m%d_%H%M')
project_dir = '/content/drive/MyDrive/yolo_training'

# log the list of datasets and zips into the project dir with the timestamp
!mkdir -p $project_dir/$timestamp && ls -l dataset zips > $project_dir/$timestamp/datasources.txt

def train_yolo():
    model = YOLO('yolov8s.pt')
    model.train(
        data='/content/dataset/data.yaml',
        epochs=100,
        batch=64,
        imgsz=640,
        patience=20,
        project=project_dir,  # <- save to Google Drive,
        name=timestamp,
        exist_ok=True,
        save=True,
        save_period=-1, # Only save best.pt and last.pt at the end of training
        val=True,
        verbose=True
    )

train_yolo()

Ultralytics 8.4.154 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2026-0916_1835, nbs=64, nms=Non

In [ ]:
import os
from ultralytics import YOLO

# Assuming 'project_dir' and 'timestamp' are available from the previous training cell execution.
# If you run this cell independently, ensure these variables are set correctly:
# project_dir = '/content/drive/MyDrive/yolo_training'
# timestamp = 'train-YYYY-MMDD_HHMM' # Replace with your actual training run folder name

# Path to the best.pt model that was just trained and saved
model_to_export_path = os.path.join(project_dir, timestamp, 'weights', 'best.pt')

print(f"Loading model from: {model_to_export_path}")

# Load the trained model
model = YOLO(model_to_export_path)

# Define the output filename for the ONNX model
onnx_filename = 'best.onnx'

# Export the model to ONNX format with specified parameters
# The 'name' parameter will ensure it's saved as 'best.onnx' in the export directory
# which by default is typically alongside the loaded model or in a run's export folder.
# Ultralytics `export` method on a loaded model usually places the output in the same directory
# as the loaded model or a logical subfolder, ensuring it's with best.pt.
model.export(
    format='onnx',
    imgsz=640,
    opset=17,
    simplify=True,
    dynamic=False,
    name=onnx_filename # Corrected from 'filename' to 'name'
)

onnx_output_dir = os.path.dirname(model_to_export_path)
print(f"ONNX model exported to: {os.path.join(onnx_output_dir, onnx_filename)}")
print("\nVerification: Listing contents of the weights directory:")
!ls -l {onnx_output_dir}

In [ ]:
import os

# Count files in train/images and train/labels
train_images_count = len(os.listdir('/content/dataset/train/images'))
train_labels_count = len(os.listdir('/content/dataset/train/labels'))

# Count files in val/images and val/labels
val_images_count = len(os.listdir('/content/dataset/val/images'))
val_labels_count = len(os.listdir('/content/dataset/val/labels'))

print(f"Train Images: {train_images_count}")
print(f"Train Labels: {train_labels_count}")
print(f"Val Images: {val_images_count}")
print(f"Val Labels: {val_labels_count}")

In [ ]:
from google.colab import runtime
runtime.unassign()